In [2]:
# 1. Clonar
!git clone https://github.com/igcondor/SCY1101_Analisis_Semestral_2026.git

# 2. Entrar a la carpeta
%cd /content/SCY1101_Analisis_Semestral_2026

# 3. Cambiarse al branch correcto ANTES de cualquier import
!git checkout condore_entrena_super

# 4. Traer cambios más recientes
!git pull

Cloning into 'SCY1101_Analisis_Semestral_2026'...
remote: Enumerating objects: 78, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 78 (delta 41), reused 19 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (78/78), 668.64 KiB | 3.89 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/SCY1101_Analisis_Semestral_2026
Branch 'condore_entrena_super' set up to track remote branch 'condore_entrena_super' from 'origin'.
Switched to a new branch 'condore_entrena_super'
Already up to date.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
import sys
sys.path.append('.')
from src.preprocesamiento_data import preprocesar, pipeline_clasificacion, pipeline_regresion
from src.model_training import entrenar_clasificacion, entrenar_regresion


SEED = 42

##Preparación de Datos


In [4]:
# Cargamos y limpiamos el dataset completo con src/preprocesamiento_data
df = preprocesar('retail_store_sales.csv')
print(f"Filas después de limpieza: {df.shape[0]}")
df.head()

Filas después de limpieza: 7983


,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Timestamp
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True,1712534400
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True,1690070400
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False,1664928000
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False,1664668800
6,TXN_3652209,CUST_07,Food,Item_1_FOOD,5.0,8.0,40.0,Credit Card,In-store,2023-06-10,True,1686355200


##Construcción de datasets para clasificación y regresión

In [5]:
#---------------CLASIFICACION-------------
#TARGET: Descuento Aplicado
X_clf_desc, y_clf_desc = pipeline_clasificacion(df) # Separacion features y target
  # Division en train (80%) y test (20%)
X_train_clf_desc, X_test_clf_desc, y_train_clf_desc, y_test_clf_desc = train_test_split(X_clf_desc, y_clf_desc, test_size=0.2, random_state=SEED)
print("TARGET: Descuento Aplicado")
print(f"Train: {X_train_clf_desc.shape[0]} filas")
print(f"Test:  {X_test_clf_desc.shape[0]} filas")
print((" "))

#-----------------REGRESION------------------
#TARGET: Total Gastado
X_reg_ts, y_reg_ts = pipeline_regresion(df) # Separacion features y target

  # Division train (80%) y test (20%)
X_train_reg_ts, X_test_reg_ts, y_train_reg_ts, y_test_reg_ts = train_test_split(X_reg_ts, y_reg_ts, test_size=0.2, random_state=SEED)
print("TARGET: Total Gastado")
print(f"Train: {X_train_reg_ts.shape[0]} filas")
print(f"Test:  {X_test_reg_ts.shape[0]} filas")
print((" "))

TARGET: Descuento Aplicado
Train: 6386 filas
Test:  1597 filas
 
TARGET: Total Gastado
Train: 6386 filas
Test:  1597 filas
 


##Modelos de Clasificacion:

### DecisionTreeClassifier
Modelo simple que divide los datos en base a un binary tree de decisiones
esto sera el modelo base contra modelos más complejos.

In [6]:
arbol_clf = DecisionTreeClassifier(random_state=SEED)
arbol_clf, y_pred_arbol = entrenar_clasificacion(
    arbol_clf, X_train_clf_desc, X_test_clf_desc,
    y_train_clf_desc, y_test_clf_desc, "Árbol de Decisión")

=== Árbol de Decisión ===
Accuracy: 0.5135
              precision    recall  f1-score   support

           0       0.50      0.52      0.51       781
           1       0.52      0.51      0.52       816

    accuracy                           0.51      1597
   macro avg       0.51      0.51      0.51      1597
weighted avg       0.51      0.51      0.51      1597



### Random Forest
Con multiples variables categoricas transformadas
con OneHotEncoder, el espacio de features es amplio y disperso. Random Forest maneja bien la dimensionalidad alta y al combinar múltiples árboles reduce
el sobreajuste que podria tener un arbol individual.

In [7]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf_clf, y_pred_rf_clf = entrenar_clasificacion(
    rf_clf, X_train_clf_desc, X_test_clf_desc,
    y_train_clf_desc, y_test_clf_desc, "Random Forest")

=== Random Forest ===
Accuracy: 0.5059
              precision    recall  f1-score   support

           0       0.50      0.53      0.51       781
           1       0.52      0.49      0.50       816

    accuracy                           0.51      1597
   macro avg       0.51      0.51      0.51      1597
weighted avg       0.51      0.51      0.51      1597



### Regresion Logistica
Este modelo es perfecto ya que estima la probabilidad de que una observación pertenezca a una de dos clases binarias. Además, es el modelo mas interpretable, permitiendo entender cuales son las variables que mas influyen en la predición.

In [8]:
lr_clf = LogisticRegression(random_state=SEED, max_iter=1000)
lr_clf, y_pred_lr_clf = entrenar_clasificacion(
    lr_clf, X_train_clf_desc, X_test_clf_desc,
    y_train_clf_desc, y_test_clf_desc, "Regresión Logística")

=== Regresión Logística ===
Accuracy: 0.5028
              precision    recall  f1-score   support

           0       0.49      0.50      0.50       781
           1       0.51      0.50      0.51       816

    accuracy                           0.50      1597
   macro avg       0.50      0.50      0.50      1597
weighted avg       0.50      0.50      0.50      1597



##Modelos de Regresion:

### Regresion Lineal
Esto es el modelo base para comparar contra modelos mas complejos. Asume una relación lineal entre los features y el target.

In [9]:
lr_reg = LinearRegression()
lr_reg, y_pred_lr_reg = entrenar_regresion(
    lr_reg, X_train_reg_ts, X_test_reg_ts,
    y_train_reg_ts, y_test_reg_ts, "Regresión Lineal")

=== Regresión Lineal ===
RMSE: 30.8209
R2:   0.8920


### Random Forest Regresion
Sirve para comparar la Regresión Lineal y evaluar si existen
relaciones no lineales entre los features y Total Spent. En el caso que Random Forest
supera significativamente a la Regresión Lineal en R2 y RMSE, entonces el problema tiene componentes no lineales que la regresion lineal no evalua.

In [10]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=SEED)
rf_reg, y_pred_rf_reg = entrenar_regresion(
    rf_reg, X_train_reg_ts, X_test_reg_ts,
    y_train_reg_ts, y_test_reg_ts, "Random Forest Regresión")

=== Random Forest Regresión ===
RMSE: 25.2501
R2:   0.9275


## Comparación Rápida de Resultados (REVISAR BIEN ESTA PARTE si corresponde aca)
Resumen de métricas básicas de los 5 modelos. El análisis profundo con validación
cruzada se realiza en el notebook 03.

In [11]:
# Clasificación
resultados_clf = pd.DataFrame({
    'Modelo': ['Árbol de Decisión', 'Random Forest', 'Regresión Logística'],
    'Accuracy': [
        accuracy_score(y_test_clf_desc, y_pred_arbol),
        accuracy_score(y_test_clf_desc, y_pred_rf_clf),
        accuracy_score(y_test_clf_desc, y_pred_lr_clf)
    ]
})
resultados_clf['Accuracy'] = resultados_clf['Accuracy'].round(4)
resultados_clf = resultados_clf.sort_values('Accuracy', ascending=False)
print("=== Clasificación ===")
print(resultados_clf.to_string(index=False))

print()

# Regresión
resultados_reg = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Random Forest'],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test_reg_ts, y_pred_lr_reg)),
        np.sqrt(mean_squared_error(y_test_reg_ts, y_pred_rf_reg))
    ],
    'R2': [
        r2_score(y_test_reg_ts, y_pred_lr_reg),
        r2_score(y_test_reg_ts, y_pred_rf_reg)
    ]
})
resultados_reg = resultados_reg.round(4)
resultados_reg = resultados_reg.sort_values('R2', ascending=False)
print("=== Regresión ===")
print(resultados_reg.to_string(index=False))

=== Clasificación ===
             Modelo  Accuracy
  Árbol de Decisión    0.5135
      Random Forest    0.5059
Regresión Logística    0.5028

=== Regresión ===
          Modelo    RMSE     R2
   Random Forest 25.2501 0.9275
Regresión Lineal 30.8209 0.8920
